# HouseholdBasketEnv — Held-out Eval + Plots (Phases 9 + 10)

After GRPO training, we:

1. **§9** Run held-out eval per task tier with the trained adapters and compare against `docs/results/baseline.json`.
2. **§10** Plot reward / KL / entropy curves from `runs/<RUN_NAME>/training_log.json`.
3. **§10.3** Surface 5 qualitative episode samples for the demo / report.

Set `RUN_NAME` to choose which run to evaluate (`main`, `ablation_a`, `ablation_b`).

> **Hardware:** Local GPU or Colab T4. The first cells clone the repo and install deps automatically.

In [ ]:
RUN_NAME = "main"   # "main" | "ablation_a" | "ablation_b"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEEDS_PER_TIER = 30
MAX_NEW_TOKENS = 96
TEMPERATURE = 0.2
LOAD_IN_4BIT = True
MAX_SEQ_LEN = 2048

import os, sys, pathlib, subprocess


CLONE_DIR = pathlib.Path("/app/hackathon/HouseholdBasketEnv")

REPO_ROOT = CLONE_DIR.resolve()
ENV_ROOT  = REPO_ROOT / "household_basket_env"
RESULTS_DIR = REPO_ROOT / "docs" / "results"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.environ["HOUSEHOLD_BASKET_PRODUCTS_PATH"] = str(ENV_ROOT / "data" / "products.json")
os.environ.setdefault("HF_HOME", str(REPO_ROOT / ".hf_cache"))

RUN_DIR = REPO_ROOT / "docs" / "runs"
ADAPTER_DIR = RUN_DIR / "adapter_final"
RESULTS_DIR = REPO_ROOT / "docs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT =", REPO_ROOT)
print("RUN_DIR   =", RUN_DIR, "exists =", RUN_DIR.exists())
print("ADAPTER   =", ADAPTER_DIR, "exists =", ADAPTER_DIR.exists())

REPO_ROOT = /app/hackathon/HouseholdBasketEnv
RUN_DIR   = /app/hackathon/HouseholdBasketEnv/docs/runs exists = True
ADAPTER   = /app/hackathon/HouseholdBasketEnv/docs/runs/adapter_final exists = True


## 1. Load model + adapters

In [2]:
model, tokenizer = None, None
model_path = str(ADAPTER_DIR) if ADAPTER_DIR.exists() else MODEL_NAME
os.environ["HTTP_PROXY"] = "http://azureproxy.jio.com:8080" 
os.environ["HTTPS_PROXY"] = "http://azureproxy.jio.com:8080"
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=LOAD_IN_4BIT,
        dtype=None,
    )
    FastLanguageModel.for_inference(model)
    print("Loaded via Unsloth.")
except Exception as e:
    print("Unsloth unavailable ->", e)

print("Model ready. Using:", model_path)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/app/hackathon/HouseholdBasketEnv/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.6.2. vLLM: 0.19.1.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.254 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 813.14it/s]


unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


[transformers] Unsloth 2026.4.8 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Loaded via Unsloth.
Model ready. Using: /app/hackathon/HouseholdBasketEnv/docs/runs/adapter_final


## 2. Held-out eval (3 tiers)

In [3]:
import json, re, statistics
from textwrap import dedent
from collections import Counter

from household_basket_env.server.environment import HouseholdBasketEnvironment
from household_basket_env.server.curriculum import load_verified_seeds

SYSTEM_PROMPT = """You are a careful nutrition-aware shopping assistant for an Indian household. \
At each step you pick exactly ONE product from the catalog and assign it to ONE household member. \
You MUST output ONLY a JSON object with three keys: \"product_id\" (string), \"member_id\" (string), \"rationale\" (1 short sentence). \
Do NOT add any prose, code fences, or extra keys."""

JSON_RE = re.compile(r"\{[^{}]*\}", re.S)

def render_observation(obs):
    members = []
    for m in obs.household:
        caps = ", ".join(f"{k}={v:.0f}" for k, v in m.thresholds_cap.items())
        members.append(f"- {m.member_id} | conds={m.conditions} | caps: {caps}")
    cands = []
    for c in obs.candidates[:50]:
        nuts = c.nutrition_per_100g
        cands.append(
            f"- {c.product_id} | {c.category} | meal={c.meal_type} | INR {c.price_inr:.0f} | "
            f"sugars_g={nuts.get('sugars_g',0):.1f} sodium_mg={nuts.get('sodium_mg',0):.0f} fat_g={nuts.get('fat_g',0):.1f}"
        )
    basket = [f"- {t.product_id} -> {t.member_id}" for t in obs.basket_so_far]
    return dedent(f"""
    Household:
    {chr(10).join(members)}

    Catalog (top 50):
    {chr(10).join(cands)}

    Basket so far ({len(basket)}):
    {chr(10).join(basket) if basket else "(empty)"}

    Budget remaining: INR {obs.budget_remaining:.0f}
    Steps used: {obs.step_index} / {obs.max_steps}

    Now pick ONE product and assign it to ONE member. Output JSON only.
    """).strip()

def chat_generate(system, user, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    inputs = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(model.device)
    out = model.generate(
        inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=temperature > 0.0,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)

def extract_json(t):
    m = JSON_RE.search(t)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

env = HouseholdBasketEnvironment()
print("Eval helpers ready.")

Eval helpers ready.


In [ ]:
def run_episode(seed, task_id):
    obs = env.reset(seed=seed, task_id=task_id)
    total = 0.0
    parse_fails = 0
    last = obs
    iter_cap = obs.max_steps * 4 + 2
    transcript = []
    for _ in range(iter_cap):
        if obs.done:
            break
        prompt = render_observation(obs)
        raw = chat_generate(SYSTEM_PROMPT, prompt)
        payload = extract_json(raw)
        if payload is None:
            parse_fails += 1
            payload = raw
        out = env.apply_raw_action(payload)
        total += out.reward or 0.0
        transcript.append({
            "step": obs.step_index,
            "raw": raw[:200],
            "parsed": payload if isinstance(payload, dict) else None,
            "reward": out.reward,
            "breakdown": out.reward_breakdown,
        })
        obs = out
        last = out
    return {
        "seed": seed,
        "task_id": task_id,
        "total_reward": round(total, 4),
        "parse_fails": parse_fails,
        "step_index": last.step_index,
        "attempt_index": last.attempt_index,
        "terminal_reward": last.terminal_reward,
        "basket_size": len(last.basket_so_far),
        "transcript": transcript,
    }

records, summary = {}, {}
for task_id in (1, 2, 3):
    payload = load_verified_seeds(task_id)
    seeds = payload["held_out_seeds"][:MAX_SEEDS_PER_TIER]
    print(f"\n=== Task {task_id}: {len(seeds)} held-out seeds ===")
    rows = [run_episode(s, task_id) for s in seeds]
    records[str(task_id)] = rows
    rewards = [r["total_reward"] for r in rows]
    term_dist = Counter(round(r["terminal_reward"], 1) for r in rows)
    summary[str(task_id)] = {
        "n_seeds": len(rows),
        "mean_reward": round(statistics.fmean(rewards), 4),
        "median_reward": round(statistics.median(rewards), 4),
        "stdev_reward": round(statistics.pstdev(rewards), 4) if len(rewards) > 1 else 0.0,
        "parse_failure_rate": round(sum(1 for r in rows if r["parse_fails"] > 0) / max(1, len(rows)), 4),
        "terminal_reward_distribution": dict(term_dist),
    }

eval_path = RESULTS_DIR / f"eval_{RUN_NAME}.json"
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump({"run_name": RUN_NAME, "summary": summary, "rows": records}, f, indent=2)
print("\nSaved ->", eval_path)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Task 1: 30 held-out seeds ===


/app/hackathon/HouseholdBasketEnv/.venv/lib64/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/app/hackathon/HouseholdBasketEnv/.venv/lib64/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/app/hackathon/HouseholdBasketEnv/.venv/lib64/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMask


=== Task 2: 30 held-out seeds ===


[transformers] Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Unsloth: Input IDs of shape torch.Size([1, 2420]) with length 2420 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
[transformers] Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

## 3. Compare against baseline

In [ ]:
baseline_path = RESULTS_DIR / "baseline.json"
if baseline_path.exists():
    with open(baseline_path) as f:
        baseline = json.load(f)
    print(f"{'Task':<6}{'Baseline':<14}{'Trained':<14}{'Δ':<10}")
    print("-" * 44)
    for task_id in ("1", "2", "3"):
        b = baseline["summary"].get(task_id, {}).get("mean_reward", float("nan"))
        t = summary[task_id]["mean_reward"]
        print(f"{task_id:<6}{b:<14.4f}{t:<14.4f}{(t-b):+.4f}")
else:
    print("No baseline.json found — run baseline_eval.ipynb first.")

## 4. Training curves (reward / KL / entropy)

In [ ]:
log_path = RUN_DIR / "training_log.json"
if log_path.exists():
    with open(log_path) as f:
        log = json.load(f)
    import matplotlib.pyplot as plt
    steps = [r["step"] for r in log if "step" in r]
    rewards = [r.get("reward", r.get("rewards/reward_basket_step", None)) for r in log]
    kl = [r.get("kl", None) for r in log]
    entropy = [r.get("entropy", r.get("completions/mean_entropy", None)) for r in log]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, ys, title in zip(axes, [rewards, kl, entropy], ["Reward", "KL", "Entropy"]):
        ys2 = [y for y in ys if y is not None]
        xs2 = steps[:len(ys2)]
        ax.plot(xs2, ys2)
        ax.set_title(title)
        ax.set_xlabel("step")
        ax.grid(True)
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / f"curves_{RUN_NAME}.png", dpi=120)
    plt.show()
    print("Saved ->", RESULTS_DIR / f"curves_{RUN_NAME}.png")
else:
    print("No training_log.json found — train first.")

## 5. Qualitative samples (5 episodes)

For the demo / report we surface 5 representative episodes: best, worst, median, an over-budget terminal, and a clean +1.0 terminal.

In [ ]:
all_rows = [r for rows in records.values() for r in rows]
sorted_rows = sorted(all_rows, key=lambda r: r["total_reward"])
samples = []
if sorted_rows:
    samples = [sorted_rows[0], sorted_rows[len(sorted_rows)//2], sorted_rows[-1]]
    full_pass = [r for r in all_rows if round(r["terminal_reward"], 1) == 1.0]
    if full_pass:
        samples.append(full_pass[0])
    over_b = [r for r in all_rows if round(r["terminal_reward"], 1) == -0.5]
    if over_b:
        samples.append(over_b[0])

samples_path = RESULTS_DIR / f"qualitative_{RUN_NAME}.json"
with open(samples_path, "w", encoding="utf-8") as f:
    json.dump(samples, f, indent=2)
print(f"Saved {len(samples)} qualitative samples -> {samples_path}")
for s in samples:
    print(f"  task={s['task_id']} seed={s['seed']} total={s['total_reward']:.3f} terminal={s['terminal_reward']} basket={s['basket_size']}")

### Outputs

- `docs/results/eval_<RUN_NAME>.json` — full per-seed eval
- `docs/results/curves_<RUN_NAME>.png` — reward / KL / entropy
- `docs/results/qualitative_<RUN_NAME>.json` — 5 representative episodes

Re-run with `RUN_NAME = "ablation_a"` and `"ablation_b"` to populate the comparison table referenced in the README.